# Brent forecasting with XGBoost

One model: gradient-boosted trees (XGBoost) that forecast the cumulative Brent futures log return
1, 5, 10, 20, 30, 40, 50 and 60 trading days ahead, one model per horizon (direct forecasts).
Forecasts are converted to prices.
Inputs: about 20 candidate predictors, reduced in every training window to one representative variable per
principal component (PCA variable selection, not PCA regression).
Benchmarks required by the brief: no-change (RW), historical mean, AR(1).

**Regimes.** Volatility and risk features (realized volatility, its short/long ratio, VIX) are inputs.
Trees can split on them first, so the effect of the other predictors can differ between calm and
turbulent markets without imposing a regime model. Section 5.3 shows how predictor importance changes
across volatility regimes.

**Sample split**

| Block | Dates | Role |
|---|---|---|
| In-sample | each fit's 1,260-day window (about 5 years) | training; the last 252 days of the window are an inner validation set for early stopping |
| Out-of-sample backtest | forecast origins 2016-01-01 to 2026-09-30 | evaluation; every forecast uses only data up to its origin |
| Live test | origins from 2026-10-01 | untouched test period; forecasts logged daily |

**Run.** `pip install xgboost`, open from `OilForecastQF`, run all cells. Results go to `Outputs/xgb/`.

In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import stats
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=RuntimeWarning)
PROJECT_ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd() / "OilForecastQF", Path.cwd().parent)
                     if (p / "Data" / "raw" / "yahoo_daily.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook from OilForecastQF (needs Data/raw/yahoo_daily.csv).")
RUN_LEAK_CHECK = False          # True: re-run a short backtest on scrambled future data (about 1 minute)
pd.set_option("display.width", 180)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#E6E8EB", "font.size": 10})

## 1. Settings

In [ ]:
@dataclass
class Config:
    data_dir: Path = PROJECT_ROOT / "Data" / "raw"
    output_dir: Path = PROJECT_ROOT / "Outputs" / "xgb"
    target: str = "brent_fut"
    horizons: tuple = (1, 5, 10, 20, 30, 40, 50, 60)   # one direct model per horizon (trading days)
    report_h: tuple = (1, 5, 10, 20, 30, 40, 50, 60)   # horizons shown in the tables
    oos_start: str = "2016-01-01"      # first out-of-sample forecast origin
    test_start: str = "2026-10-01"     # live test period
    window: int = 1260                 # rolling training window (trading days, about 5 years)
    refit_every: int = 21              # re-train every 21 trading days (about monthly)
    inner_val: int = 252               # last 252 days of each window: early-stopping validation
    ffill_limit: int = 3
    fred_lag: int = 1                  # FRED daily series enter one observation late
    # XGBoost: shallow trees, slow learning, row/column subsampling, strong L2 penalty
    n_estimators: int = 500
    learning_rate: float = 0.03
    max_depth: int = 2
    min_child_weight: float = 50
    subsample: float = 0.7
    colsample_bytree: float = 0.7
    reg_lambda: float = 10.0
    early_stopping: int = 50
    seed: int = 7
    shrink: float = 0.5                # final forecast = 0.5 x tree forecast (equal-weight combination with no change)
    use_pca: bool = True               # False: train on all candidate features (comparison in section 5.4)
    pca_min_eig: float = 1.0           # PCA variable selection: keep components with eigenvalue > 1 (Kaiser)
    use_season: bool = False           # day-of-year features off: they overfit in a 5-year window
    plot_h: int = 20
    path_years: int = 4                # forecast-path chart: last 4 years of out-of-sample origins
    cost_bps: float = 5.0
    target_vol: float = 25.0
    max_leverage: float = 2.0
    trade_h: int = 20                  # trading rule: use the 20-day forecast and hold for 20 days


cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

## 2. Data and features

Sources: `Data/raw/yahoo_daily.csv`; `Data/raw/fred_daily.csv` if present (lagged one observation).
The calendar is the Brent trading calendar; other series are forward-filled for at most 3 days;
non-positive prices are set to missing. XGBoost handles missing values itself, so no imputation.

About 20 candidate predictors, chosen on economic grounds. Section 2b then keeps a subset of these original
variables with PCA-based selection. All are known at the close of day $t$.

**Every feature is a change or a z-score, never a raw level.** A tree cannot extrapolate: if a level
(VIX, OVX, the yield spread) moves outside the range of its training window, as in 2020 and 2022, the
tree treats it like the most extreme value it has seen, and the forecast is driven by a handful of old days.
A z-score measures the value against its own past year: $z_t=(v_t-	ext{mean}_{252})/	ext{sd}_{252}$.

| Block | Features | Idea |
|---|---|---|
| Brent momentum and trend | log returns over 1, 5, 20, 60 days; log gap to 50- and 200-day averages | momentum, reversal, trend |
| Volatility (regime) | 20/60-day realized volatility ratio; z-scores of 60-day realized volatility and VIX | calm vs turbulent market |
| Oil market | z-scores of Brent–WTI spread and 3-2-1 crack spread; crack spread 20-day change | regional balance, refining margin |
| Cross-market (20-day log return) | natural gas, XLE, S&P 500, dollar index, gold, copper, and USD/CAD, USD/NOK, EEM if present | energy equities, risk appetite, dollar, industrial demand |
| Macro (FRED, if present) | 10Y–3M spread 60-day change; OVX z-score; 10Y breakeven 20-day change | business cycle, oil risk, inflation |

Seasonality (`use_season`) is off: in a 5-year window each calendar date appears only five times, and in the
first run the day-of-year terms ranked among the top features while the out-of-sample error rose.

In [ ]:
def load_data(cfg):
    raw = pd.read_csv(cfg.data_dir / "yahoo_daily.csv", parse_dates=["date"]).set_index("date").sort_index()
    raw = raw[~raw.index.duplicated(keep="last")].select_dtypes("number")
    raw = raw.where(raw > 0)
    price = raw[cfg.target].dropna()
    other = raw.drop(columns=[cfg.target])
    fred_path = cfg.data_dir / "fred_daily.csv"
    if fred_path.is_file():
        fred = pd.read_csv(fred_path, parse_dates=["date"]).set_index("date").sort_index().select_dtypes("number")
        keep = [c for c in ("t10y3m", "ovx", "breakeven10y") if c in fred]
        fred = fred[keep].dropna(how="all").shift(cfg.fred_lag)       # published the next day
        other = other.join(fred, how="outer")
    other = other.reindex(other.index.union(price.index)).ffill(limit=cfg.ffill_limit).reindex(price.index)
    return price, other


def zscore(s, n=252):
    roll = s.rolling(n, min_periods=int(0.8 * n))
    return (s - roll.mean()) / roll.std()


def build_features(price, other, cfg):
    lp = np.log(price)
    r = lp.diff()
    vol = pd.DataFrame({"rv20": r.rolling(20, min_periods=15).std() * np.sqrt(252),
                        "rv60": r.rolling(60, min_periods=45).std() * np.sqrt(252)})
    f = {f"brent_ret{k}": lp.diff(k) for k in (1, 5, 20, 60)}
    f["brent_gap_ma50"] = lp - np.log(price.rolling(50, min_periods=40).mean())
    f["brent_gap_ma200"] = lp - np.log(price.rolling(200, min_periods=160).mean())
    f["rv_ratio"] = vol["rv20"] / vol["rv60"]
    f["rv60_z"] = zscore(vol["rv60"])
    if "vix" in other:
        f["vix_z"] = zscore(other["vix"])
    if "wti_fut" in other:
        f["brent_wti_z"] = zscore(lp - np.log(other["wti_fut"]))
    if {"gasoline", "heating_oil", "wti_fut"} <= set(other.columns):
        cl = other["wti_fut"]
        crack = ((2 * 42 * other["gasoline"] + 42 * other["heating_oil"] - 3 * cl) / 3 / cl).where(lambda s: s.abs() < 2)
        f["crack_z"], f["crack_d20"] = zscore(crack), crack.diff(20)
    for c in ("natgas", "xle", "sp500", "dxy", "gold", "copper", "usdcad", "usdnok", "eem"):
        if c in other and other[c].notna().sum() > 500:
            f[f"{c}_ret20"] = np.log(other[c]).diff(20)
    if "t10y3m" in other:
        f["t10y3m_d60"] = other["t10y3m"].diff(60)
    if "ovx" in other:
        f["ovx_z"] = zscore(other["ovx"])
    if "breakeven10y" in other:
        f["breakeven_d20"] = other["breakeven10y"].diff(20)
    if cfg.use_season:
        doy = price.index.dayofyear.to_numpy()
        f["season_sin"] = pd.Series(np.sin(2 * np.pi * doy / 365.25), index=price.index)
        f["season_cos"] = pd.Series(np.cos(2 * np.pi * doy / 365.25), index=price.index)
    X = pd.DataFrame(f, index=price.index).replace([np.inf, -np.inf], np.nan)
    Y = {h: 100 * (lp.shift(-h) - lp) for h in cfg.horizons}         # cumulative h-day log return, %
    return X, Y, lp, vol


price, other = load_data(cfg)
X, Y, logp, VOL = build_features(price, other, cfg)
pd.concat([price.rename("brent_fut"), X], axis=1).rename_axis("date").to_csv(cfg.output_dir / "model_panel.csv")
display(pd.DataFrame({"first date": [price.index[0].date()], "last date": [price.index[-1].date()],
                      "Brent observations": [len(price)], "number of features": [X.shape[1]]}).T.rename(columns={0: ""}))
display(Markdown("**Features:** " + ", ".join(f"`{c}`" for c in X.columns)))
display(X.tail(3).T.round(4))

## 2b. PCA variable selection

Many candidate features measure overlapping things (risk appetite, trend, the dollar). PCA is used only to
**choose which original variables to keep**; the model is then trained on those original variables, not on
principal components (this is not a PCA regression).

Method (Jolliffe 1972, method B4), on the training window only:

1. Standardize the candidate features and compute the eigenvalues and loadings of their correlation matrix.
2. Keep the components with eigenvalue above 1 (Kaiser rule): each kept component explains more variance than a single original variable.
3. Go through the kept components in order; for each, select the original variable with the largest absolute loading that has not been selected yet.

The result is one representative variable per independent direction of variation, with redundant variables dropped.
The selection uses only the features, never the target, so it cannot pick variables by their past forecasting success.
It is repeated inside every rolling window, so the set can change over time. Below: the latest window.

In [ ]:
def pca_select(A, names, min_eig=1.0):
    """Jolliffe B4: one original variable per retained principal component."""
    D = pd.DataFrame(A, columns=names)
    D = D.loc[:, D.notna().mean() > 0.8]                       # ignore features mostly missing in the window
    D = D.fillna(D.mean())
    Z = (D - D.mean()) / D.std().replace(0, np.nan)
    Z = Z.dropna(axis=1)
    eig, vec = np.linalg.eigh(np.corrcoef(Z.to_numpy().T))
    order = np.argsort(eig)[::-1]
    eig, vec = eig[order], vec[:, order]
    k = max(1, int((eig > min_eig).sum()))
    load = pd.DataFrame(vec[:, :k], index=Z.columns, columns=[f"PC{j + 1}" for j in range(k)])
    chosen = []
    for j in range(k):
        for v in load.iloc[:, j].abs().sort_values(ascending=False).index:
            if v not in chosen:
                chosen.append(v)
                break
    sel = [list(names).index(v) for v in chosen]
    return sel, eig, load


Nw = len(X)
sel_now, eig_now, load_now = pca_select(X.to_numpy(np.float32)[Nw - cfg.window:Nw], X.columns, cfg.pca_min_eig)
k_now = load_now.shape[1]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].bar(np.arange(1, len(eig_now) + 1), eig_now, color=["#1F7A5A" if e > cfg.pca_min_eig else "#C8CDD2" for e in eig_now])
axes[0].axhline(cfg.pca_min_eig, color="black", ls="--", lw=0.8)
axes[0].set_title(f"Eigenvalues, latest window: {k_now} components kept "
                  f"({100 * eig_now[:k_now].sum() / eig_now.sum():.0f}% of variance)", fontsize=10)
axes[0].set_xlabel("principal component")
im = axes[1].imshow(load_now.to_numpy(), cmap="RdBu_r", vmin=-0.7, vmax=0.7, aspect="auto")
axes[1].set_yticks(range(len(load_now))); axes[1].set_yticklabels(load_now.index, fontsize=8)
axes[1].set_xticks(range(k_now)); axes[1].set_xticklabels(load_now.columns)
for j, i in enumerate([list(load_now.index).index(X.columns[c]) for c in sel_now]):
    axes[1].add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=2))
axes[1].set_title("Loadings of the kept components (boxed: variable selected for that component)", fontsize=10)
plt.colorbar(im, ax=axes[1], fraction=0.03)
fig.tight_layout(); fig.savefig(cfg.output_dir / "pca_selection.png", dpi=150); plt.show()
display(pd.DataFrame({"component": load_now.columns, "eigenvalue": eig_now[:k_now].round(2),
                      "selected variable": [X.columns[c] for c in sel_now],
                      "loading": [load_now.loc[X.columns[c], pc].round(2) for c, pc in zip(sel_now, load_now.columns)]}))

## 3. Model

For each horizon $h$, a separate model (direct multi-step forecast):

$$y_{t,h} = 100\,(\log P_{t+h}-\log P_t) = F_h(x_t) + e_{t,h},\qquad F_h(x)=\sum_{m=1}^{M} \eta\, f_m(x)$$

where each $f_m$ is a regression tree of depth 2 fitted to the residuals of the previous trees.

| Setting | Value | Why |
|---|---|---|
| Tree depth | 2 | each tree uses at most two variables: main effects and pairwise interactions such as "copper matters only when volatility is high" |
| Learning rate $\eta$ | 0.03 | small steps |
| Trees $M$ | up to 500, early stopping after 50 rounds without improvement on the inner validation set | the data choose how complex the model gets |
| Minimum observations per leaf | 50 | no leaf built on a handful of days |
| Row / column subsampling | 70% / 70% | less overfitting, less dependence on any one variable |
| L2 penalty on leaf values | 10 | shrinks forecasts toward zero |
| Starting value | 0 | with no signal, the forecast equals the no-change forecast |
| Final forecast | $\hat y_{t,h} = 0.5\,F_h(x_t)$ | equal-weight combination with the no-change forecast |

**Why shrink by half.** Oil returns are very noisy, so even a useful model makes large errors when it takes a
strong view. Averaging with the no-change forecast keeps the direction of the signal and halves its size.
With $f$ the tree forecast and $y$ the outcome, $\text{MSE}(\lambda f)=\text{MSE}(0)-2\lambda E[fy]+\lambda^2E[f^2]$:
when the signal is weak, a smaller $\lambda$ cuts the error from noise faster than it loses signal.
The unshrunk forecast is reported as `XGB_raw` for comparison.

Hyperparameters and the shrink weight are fixed in advance and not tuned on the backtest.

In [ ]:
def make_model(cfg, n_estimators=None):
    return xgb.XGBRegressor(n_estimators=n_estimators or cfg.n_estimators, learning_rate=cfg.learning_rate,
                            max_depth=cfg.max_depth, min_child_weight=cfg.min_child_weight,
                            subsample=cfg.subsample, colsample_bytree=cfg.colsample_bytree,
                            reg_lambda=cfg.reg_lambda, base_score=0.0, tree_method="hist",
                            objective="reg:squarederror", random_state=cfg.seed, n_jobs=4,
                            early_stopping_rounds=cfg.early_stopping)


def fit_one(A, y, scale, i0, h, cfg):
    """Train for origin i0: rows s with s + h <= i0 (targets already observed).
    Split: fit rows | gap of h days | last `inner_val` rows for early stopping.
    Interval: quantiles of errors / scale over the whole window, scale = 20-day vol over h days."""
    lo, hi = i0 - cfg.window, i0 - h + 1                       # rows lo..hi-1
    v0 = hi - cfg.inner_val
    fit_rows = np.arange(lo, v0 - h)
    val_rows = np.arange(v0, hi)
    fit_rows = fit_rows[np.isfinite(y[fit_rows])]
    val_rows = val_rows[np.isfinite(y[val_rows])]
    m = make_model(cfg)
    m.fit(A[fit_rows], y[fit_rows], eval_set=[(A[val_rows], y[val_rows])], verbose=False)
    rows = np.r_[fit_rows, val_rows]
    z = (y[rows] - cfg.shrink * m.predict(A[rows])) / scale[rows]
    q = np.nanquantile(z, [0.025, 0.10, 0.90, 0.975])          # standardized error quantiles
    return m, q, fit_rows, val_rows

## 4. Rolling training and forecasts

- **Rolling window:** each model is trained on the previous 1,260 trading days (about 5 years). For horizon $h$, the last training row is $t-h$, so every target is already observed at the forecast date.
- **Inner validation:** inside each window, the last 252 days are held out (with an $h$-day gap) to decide when to stop adding trees. This is the training / validation split; it is repeated in every window and never touches data after the forecast date.
- **Intervals:** forecast errors in the window are divided by the 20-day realized volatility scaled to $h$ days; the 2.5/10/90/97.5% quantiles of these standardized errors are multiplied by today's volatility. Intervals therefore widen in turbulent periods.
- **Re-training:** every 21 trading days (about monthly) from 2016-01-01. Between re-trainings the model is fixed and forecasts every day with that day's features, so new daily data enter immediately.
- **Benchmarks** on the same dates and window: RW (no change), HistMean (window mean h-day return), AR(1) on daily returns, iterated.
- **In-sample forecasts:** the latest model of each horizon applied to its own training rows.

In [ ]:
def run_backtest(X, Y, lp, vol, cfg, verbose=True):
    dates, N = X.index, len(X)
    A = X.to_numpy(np.float32)
    rr = 100 * lp.diff().to_numpy()
    rv = vol["rv20"].to_numpy()
    start = max(int(dates.searchsorted(pd.Timestamp(cfg.oos_start))), cfg.window + 200)
    refits = list(range(start, N, cfg.refit_every))
    out = {h: {k: np.full(N, np.nan) for k in ("XGB", "XGB_raw", "q025", "q10", "q90", "q975", "HistMean", "AR")}
           for h in cfg.horizons}
    imp, ntrees, ins, sel_log, t0 = [], [], {}, [], time.time()
    for k, i0 in enumerate(refits):
        i1 = refits[k + 1] if k + 1 < len(refits) else N
        sel = (pca_select(A[i0 - cfg.window + 1:i0 + 1], X.columns, cfg.pca_min_eig)[0]   # features known at i0
               if cfg.use_pca else list(range(X.shape[1])))
        As = A[:, sel]
        sel_log.append({"refit": dates[i0], **{X.columns[c]: 1 for c in sel}})
        w = rr[i0 - cfg.window + 1:i0 + 1]
        c, phi = np.polynomial.polynomial.polyfit(w[:-1], w[1:], 1)
        for h in cfg.horizons:
            y = Y[h].to_numpy()
            scale = 100 * rv * np.sqrt(h / 252)
            m, q, fit_rows, val_rows = fit_one(As, y, scale, i0, h, cfg)
            o = out[h]
            o["XGB_raw"][i0:i1] = m.predict(As[i0:i1])
            o["XGB"][i0:i1] = cfg.shrink * o["XGB_raw"][i0:i1]
            for name, qq in zip(("q025", "q10", "q90", "q975"), q):
                o[name][i0:i1] = qq * scale[i0:i1]
            o["HistMean"][i0:i1] = np.nanmean(y[i0 - cfg.window:i0 - h + 1])
            cur, acc = rr[i0:i1].copy(), np.zeros(i1 - i0)
            for _ in range(h):
                cur = c + phi * cur
                acc = acc + cur
            o["AR"][i0:i1] = acc
            gain = m.get_booster().get_score(importance_type="total_gain")
            imp.append({"refit": dates[i0], "h": h, **{X.columns[sel[int(f[1:])]]: g for f, g in gain.items()}})
            ntrees.append({"refit": dates[i0], "h": h, "trees": m.best_iteration + 1})
            if k == len(refits) - 1:
                ins[h] = {"model": m, "rows": fit_rows, "q": q, "scale": scale, "val_rows": val_rows, "sel": sel}
    if verbose:
        print(f"{len(refits)} re-trainings x {len(cfg.horizons)} horizons, {time.time() - t0:.0f}s. "
              f"First origin {dates[start].date()}, last {dates[-1].date()}.")
    imp = pd.DataFrame(imp).fillna(0.0)
    sel_log = pd.DataFrame(sel_log).fillna(0).set_index("refit").reindex(columns=X.columns, fill_value=0)
    return out, imp, pd.DataFrame(ntrees), ins, sel_log


out, importance, ntrees, ins, sel_log = run_backtest(X, Y, logp, VOL, cfg)
ntrees.to_csv(cfg.output_dir / "trees_per_fit.csv", index=False)
sel_log.to_csv(cfg.output_dir / "pca_selection_log.csv")
freq = (100 * sel_log.mean()).sort_values(ascending=False)
ax = freq.plot.bar(figsize=(11, 3.2), color="#1F7A5A")
ax.set_ylabel("% of re-trainings"); ax.set_ylim(0, 100)
ax.set_title(f"How often PCA selection kept each variable ({len(sel_log)} re-trainings; "
             f"median {int(sel_log.sum(axis=1).median())} variables per window)", fontsize=10)
plt.tight_layout(); plt.savefig(cfg.output_dir / "pca_selection_frequency.png", dpi=150); plt.show()
display(Markdown("**Trees kept by early stopping** (median over re-trainings)"))
display(ntrees[ntrees["h"].isin(cfg.report_h)].groupby("h")["trees"].describe()[["min", "50%", "max"]].T.round(0))

In [ ]:
def to_long(cols_by_h, lp, rows_by_h, sample=None):
    dates, N, P, L = lp.index, len(lp), np.exp(lp.to_numpy()), lp.to_numpy()
    recs = []
    for h, rows in rows_by_h.items():
        t = np.asarray(rows)
        tgt = t + h
        has = tgt < N
        d = pd.DataFrame({"origin": dates[t], "h": h,
                          "target_date": [dates[j] if j < N else pd.NaT for j in tgt], "P_t": P[t],
                          "y": np.where(has, L[np.minimum(tgt, N - 1)] - L[t], np.nan), "RW": 0.0})
        for name, arr in cols_by_h[h].items():
            d[name] = arr[t] / 100
        recs.append(d)
    fc = pd.concat(recs, ignore_index=True)
    fc["P_target"] = fc["P_t"] * np.exp(fc["y"])
    test = pd.Timestamp(cfg.test_start)
    fc["sample"] = sample or np.where(fc["origin"] >= test, "Live test",
                                      np.where(fc["target_date"] < test, "Out-of-sample", "straddles test start"))
    return fc


rows_oos = {h: np.flatnonzero(np.isfinite(out[h]["XGB"])) for h in cfg.horizons}
fc = to_long(out, logp, rows_oos)
A_all = X.to_numpy(np.float32)
ins_cols, ins_rows = {}, {}
for h, d in ins.items():
    arr = {k: np.full(len(X), np.nan) for k in ("XGB", "XGB_raw", "q025", "q10", "q90", "q975")}
    arr["XGB_raw"][d["rows"]] = d["model"].predict(A_all[d["rows"]][:, d["sel"]])
    arr["XGB"][d["rows"]] = cfg.shrink * arr["XGB_raw"][d["rows"]]
    for name, v in zip(("q025", "q10", "q90", "q975"), d["q"]):
        arr[name][d["rows"]] = v * d["scale"][d["rows"]]
    ins_cols[h], ins_rows[h] = arr, d["rows"]
fc_in = to_long(ins_cols, logp, ins_rows, sample="In-sample")
fc.to_csv(cfg.output_dir / "forecasts_out_of_sample.csv", index=False)
fc_in.to_csv(cfg.output_dir / "forecasts_in_sample.csv", index=False)
r20 = ins[cfg.plot_h]["rows"]
print(f"In-sample rows of the latest {cfg.plot_h}-day model: {X.index[r20[0]].date()} to {X.index[r20[-1]].date()}")
display(fc.groupby("sample")["origin"].agg(["min", "max", "count"]).rename(columns={"count": "rows (origins x horizons)"}))

## 5. Results

### 5.1 In-sample vs out-of-sample accuracy

Price errors in \$/bbl. **RMSE ratio vs RW** below 1 means the model beats the no-change forecast.
DM p-value: one-sided Diebold–Mariano test (Harvey–Leybourne–Newbold correction, Newey–West variance) that the model has lower squared price error than RW.
Direction: hit rate, and precision / recall / F1 for "price goes up"; `up_share_%` is the share of up moves, the precision a model gets by always saying "up". Coverage: share of outcomes inside the 80% and 95% intervals.

In [ ]:
def dm_pvalue(e_m, e_b, h):
    d = e_m ** 2 - e_b ** 2
    n = len(d)
    if n < 30 or np.allclose(d, 0):
        return np.nan
    u = d - d.mean()
    v = u @ u / n + 2 * sum((1 - k / h) * (u[k:] @ u[:-k]) / n for k in range(1, h))
    if v <= 0:
        return np.nan
    stat = d.mean() / np.sqrt(v / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return stats.t.cdf(stat, n - 1)


def evaluate(df, models):
    rows = []
    for (h, smp), g in df.dropna(subset=["y"]).groupby(["h", "sample"]):
        e_rw = g["P_t"] - g["P_target"]
        for m in models:
            if m not in g or g[m].isna().all():
                continue
            e = g["P_t"] * np.exp(g[m]) - g["P_target"]
            row = {"sample": smp, "h": h, "model": m, "n": len(g), "MAE_$": e.abs().mean(),
                   "RMSE_$": np.sqrt((e ** 2).mean()), "MAPE_%": 100 * (e.abs() / g["P_target"]).mean(),
                   "RMSE_ratio_vs_RW": np.sqrt((e ** 2).mean() / (e_rw ** 2).mean())}
            if m != "RW":
                row["DM_p"] = dm_pvalue(e.to_numpy(), e_rw.to_numpy(), h)
                up_hat, up = g[m] > 0, g["y"] > 0
                tp = (up_hat & up).sum()
                prec, rec = tp / max(up_hat.sum(), 1), tp / max(up.sum(), 1)
                row.update({"hit_%": 100 * (up_hat == up).mean(), "up_share_%": 100 * up.mean(),
                            "precision_up": prec, "recall_up": rec,
                            "F1_up": 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0})
            if m == "XGB":
                res = g["y"] - g[m]
                row["cover80_%"] = 100 * ((res >= g["q10"]) & (res <= g["q90"])).mean()
                row["cover95_%"] = 100 * ((res >= g["q025"]) & (res <= g["q975"])).mean()
            rows.append(row)
    return pd.DataFrame(rows)


MODELS = ["RW", "HistMean", "AR", "XGB_raw", "XGB"]
met = evaluate(pd.concat([fc_in, fc[fc["sample"].isin(["Out-of-sample", "Live test"])]]), MODELS)
met.to_csv(cfg.output_dir / "metrics.csv", index=False)
xm = met[met["model"] == "XGB"]
order = [c for c in ["In-sample", "Out-of-sample", "Live test"] if c in set(xm["sample"])]
summary = pd.concat({"RMSE ratio vs RW": xm.pivot(index="h", columns="sample", values="RMSE_ratio_vs_RW")[order],
                     "hit rate %": xm.pivot(index="h", columns="sample", values="hit_%")[order]}, axis=1)
summary = summary.reindex(list(cfg.report_h)).round(3)
display(Markdown("**XGBoost: in-sample vs out-of-sample (price RMSE ratio vs RW, direction hit rate)**"))
display(summary)

In [ ]:
oos = met[met["sample"] == "Out-of-sample"]
display(Markdown("**Out-of-sample, all models: price RMSE ratio vs RW**"))
display(oos.pivot(index="model", columns="h", values="RMSE_ratio_vs_RW").loc[MODELS, list(cfg.report_h)].round(3))
display(Markdown("**Out-of-sample, XGBoost in detail**"))
cols = ["h", "n", "MAE_$", "RMSE_$", "MAPE_%", "RMSE_ratio_vs_RW", "DM_p", "hit_%", "up_share_%", "precision_up",
        "recall_up", "F1_up", "cover80_%", "cover95_%"]
display(oos[(oos["model"] == "XGB") & oos["h"].isin(cfg.report_h)][cols].round(3).set_index("h"))

### 5.2 What the forecasts look like: in-sample and out-of-sample

**Forecast paths.** Every 21 trading days, the 1-to-60-day forecast path is drawn from that day's price
(green: model expects a rise over 60 days; red: a fall). Black is the realized price.
Top: in-sample, the latest model on its own training rows. Bottom: out-of-sample, the last few years of rolling forecasts; the live test period is shaded.

**Does the forecast sort the outcomes?** Forecasts are split into five equal groups, from the most bearish (Q1) to the most bullish (Q5).
If the model has signal, the realized return rises from Q1 to Q5. IC is the rank correlation between forecast and outcome.

In [ ]:
def paths(df, origins):
    piv = df.pivot_table(index="origin", columns="h", values="XGB")
    p0 = df.drop_duplicates("origin").set_index("origin")["P_t"]
    for o in origins:
        if o not in piv.index or piv.loc[o].isna().any():
            continue
        xs = [o] + [o + pd.offsets.BDay(int(h)) for h in piv.columns]
        ys = [p0[o]] + list(p0[o] * np.exp(piv.loc[o].to_numpy()))
        yield xs, ys, ys[-1] > ys[0]


oos_fc = fc[fc["sample"].isin(["Out-of-sample", "Live test", "straddles test start"])]
o_orig = np.sort(oos_fc["origin"].unique())
o_orig = o_orig[o_orig >= o_orig[-1] - np.timedelta64(365 * cfg.path_years, "D")][::-1][::21][::-1]
i_orig = np.sort(fc_in.groupby("origin")["h"].nunique().loc[lambda s: s == len(cfg.horizons)].index)[::21]

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5))
for ax, df, origins, title in [
        (axes[0], fc_in, i_orig, "In-sample: forecast paths of the latest model on its training period"),
        (axes[1], oos_fc, o_orig, f"Out-of-sample: rolling forecast paths, last {cfg.path_years} years")]:
    lo_d, hi_d = pd.Timestamp(origins[0]), pd.Timestamp(origins[-1]) + pd.offsets.BDay(60)
    act = price.loc[lo_d:hi_d]
    ax.plot(act.index, act, color="black", lw=1.0, label="actual price")
    for xs, ys, up in paths(df, origins):
        ax.plot(xs, ys, color="#1F9D55" if up else "#C0392B", lw=1.3, alpha=0.9)
        ax.plot(xs[0], ys[0], "o", color="#1F9D55" if up else "#C0392B", ms=2.5)
    if df is oos_fc:
        ax.axvspan(pd.Timestamp(cfg.test_start), hi_d, color="#C38728", alpha=0.12, label="live test")
    ax.plot([], [], color="#1F9D55", label="forecast path: up"); ax.plot([], [], color="#C0392B", label="forecast path: down")
    ax.set_title(title, fontsize=11); ax.set_ylabel("$/bbl"); ax.legend(fontsize=8, loc="upper left")
fig.tight_layout(); fig.savefig(cfg.output_dir / "forecast_paths.png", dpi=150); plt.show()

h = cfg.plot_h
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
ic_rows = []
for ax, smp in zip(axes, ["In-sample", "Out-of-sample"]):
    src = fc_in if smp == "In-sample" else fc[fc["sample"] == "Out-of-sample"]
    g = src[src["h"] == h].dropna(subset=["y", "XGB"]).copy()
    g["Q"] = pd.qcut(g["XGB"].rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)])
    b = g.groupby("Q", observed=True).agg(forecast=("XGB", "mean"), realized=("y", "mean")) * 100
    b.plot.bar(ax=ax, color=["#A7D3C2", "#1F7A5A"], rot=0)
    ic = stats.spearmanr(g["XGB"], g["y"]).correlation
    ic_rows.append({"sample": smp, "h": h, "IC": ic, "Q5 - Q1 realized (%)": b["realized"].iloc[-1] - b["realized"].iloc[0]})
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{smp}: {h}-day return by forecast quintile (IC = {ic:.2f})", fontsize=10)
    ax.set_xlabel("forecast quintile (Q1 most bearish, Q5 most bullish)"); ax.set_ylabel("mean return, %")
fig.tight_layout(); fig.savefig(cfg.output_dir / "forecast_quintiles.png", dpi=150); plt.show()
display(pd.DataFrame(ic_rows).round(3))

fig, ax = plt.subplots(figsize=(11, 3.8))
allh = met.pivot_table(index="h", columns=["sample", "model"], values="RMSE_ratio_vs_RW")
for (smp, mdl), col_, ls in [(("In-sample", "XGB"), "#A7D3C2", "-"), (("Out-of-sample", "XGB"), "#1F7A5A", "-"),
                             (("Out-of-sample", "HistMean"), "#888888", "--"), (("Out-of-sample", "AR"), "#C38728", "--")]:
    if (smp, mdl) in allh:
        ax.plot(allh.index, allh[(smp, mdl)], color=col_, ls=ls, lw=1.6 if mdl == "XGB" else 1.0, marker="o", ms=4,
                label=f"{mdl}, {smp.lower()}")
ax.axhline(1, color="black", lw=0.8)
ax.set_xlabel("horizon (trading days)"); ax.set_ylabel("price RMSE / RW")
ax.set_xticks(list(cfg.horizons))
ax.set_title("Price RMSE relative to no change, by horizon (below 1 = better)", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(cfg.output_dir / "rmse_ratio_by_horizon.png", dpi=150); plt.show()

ratio = xm.pivot(index="h", columns="sample", values="RMSE_ratio_vs_RW")[order].reindex(list(cfg.report_h))
ax = ratio.plot.bar(figsize=(8, 3.5), color=["#A7D3C2", "#1F7A5A", "#C38728"][:len(order)], rot=0)
ax.axhline(1, color="black", lw=0.8)
ax.set_ylim(min(0.85, np.nanmin(ratio.values) - 0.02), max(1.1, np.nanmax(ratio.values) + 0.02))
ax.set_title("XGBoost price RMSE / RW price RMSE (below 1 = better than no change)", fontsize=11)
ax.set_xlabel("horizon (trading days)")
plt.tight_layout(); plt.savefig(cfg.output_dir / "rmse_ratio_in_vs_out.png", dpi=150); plt.show()

### 5.3 What drives the forecast, and does it change with the regime?

Left: average share of total gain by feature over all re-trainings (20-day model).
Right: SHAP values of the latest 20-day model, the contribution of each feature to each forecast in % return.
The table splits mean |SHAP| between high- and low-volatility days (20-day realized volatility above or below
its median in the window). Different rankings mean the model uses different predictors in the two regimes.

In [ ]:
imp_h = importance[importance["h"] == cfg.plot_h].drop(columns=["refit", "h"])
share = imp_h.div(imp_h.sum(axis=1), axis=0).mean().sort_values(ascending=False)
share.to_csv(cfg.output_dir / f"importance_h{cfg.plot_h}.csv")

d = ins[cfg.plot_h]
rows = d["rows"]
contrib = d["model"].get_booster().predict(xgb.DMatrix(A_all[rows][:, d["sel"]]), pred_contribs=True)[:, :-1]
shap = pd.DataFrame(contrib, columns=X.columns[d["sel"]], index=X.index[rows])
rv = VOL["rv20"].iloc[rows]
high = rv > rv.median()
by_regime = pd.DataFrame({"high volatility": shap[high.to_numpy()].abs().mean(),
                          "low volatility": shap[~high.to_numpy()].abs().mean()})
by_regime["all days"] = shap.abs().mean()
by_regime = by_regime.sort_values("all days", ascending=False)
by_regime.to_csv(cfg.output_dir / f"shap_by_regime_h{cfg.plot_h}.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
share.head(15)[::-1].plot.barh(ax=axes[0], color="#1F7A5A")
axes[0].set_title(f"Share of total gain, {cfg.plot_h}-day model (average over re-trainings)", fontsize=10)
by_regime.head(12)[["low volatility", "high volatility"]][::-1].plot.barh(ax=axes[1], color=["#A7D3C2", "#C0392B"])
axes[1].set_title(f"Mean |SHAP| (% return), latest {cfg.plot_h}-day model, by volatility regime", fontsize=10)
fig.tight_layout(); fig.savefig(cfg.output_dir / "importance.png", dpi=150); plt.show()
display(by_regime.head(12).round(3))

### 5.4 Robustness

Out-of-sample price RMSE ratio vs RW, split by volatility regime at the forecast origin and by year.
High volatility: 60-day realized volatility above the median of all its earlier values (known at the origin).

In [ ]:
o = fc[fc["sample"] == "Out-of-sample"].dropna(subset=["y"]).copy()
rv60 = VOL["rv60"]
vol_regime = pd.Series(np.where(rv60 > rv60.expanding(250).median().shift(1), "high volatility", "low volatility"),
                       index=X.index).where(rv60.notna())
o["regime"], o["year"] = o["origin"].map(vol_regime), o["origin"].dt.year


def ratio_by(df, key):
    f = lambda g: np.sqrt(((g["P_t"] * np.exp(g["XGB"]) - g["P_target"]) ** 2).mean()
                          / ((g["P_t"] - g["P_target"]) ** 2).mean())
    df = df[df["h"].isin(cfg.report_h)]
    return df.groupby([key, "h"])[["P_t", "XGB", "P_target"]].apply(f).unstack("h").round(3)


rob_reg, rob_year = ratio_by(o, "regime"), ratio_by(o, "year")
rob_reg.to_csv(cfg.output_dir / "robustness_regime.csv"); rob_year.to_csv(cfg.output_dir / "robustness_year.csv")
display(Markdown("**By volatility regime**")); display(rob_reg)
display(Markdown("**By year**")); display(rob_year)

**Feature selection: PCA subset vs all candidates.** The same backtest (same window, re-training dates, model settings and shrinkage)
is re-run with the other feature setting. Out-of-sample price RMSE ratio vs RW and the 20-day IC.

In [ ]:
alt_cfg = replace(cfg, use_pca=not cfg.use_pca, horizons=cfg.report_h)     # comparison on the table horizons
out_alt = run_backtest(X, Y, logp, VOL, alt_cfg, verbose=False)[0]
fc_alt = to_long({h: {"XGB": out_alt[h]["XGB"]} for h in alt_cfg.horizons}, logp,
                 {h: np.flatnonzero(np.isfinite(out_alt[h]["XGB"])) for h in alt_cfg.horizons})


def oos_summary(df):
    g = df[(df["sample"] == "Out-of-sample") & df["h"].isin(cfg.report_h)].dropna(subset=["y"])
    rmse = g.groupby("h").apply(lambda d: np.sqrt(((d["P_t"] * np.exp(d["XGB"]) - d["P_target"]) ** 2).mean()
                                                 / ((d["P_t"] - d["P_target"]) ** 2).mean()), include_groups=False)
    ic = g.groupby("h").apply(lambda d: stats.spearmanr(d["XGB"], d["y"]).correlation, include_groups=False)
    return rmse, ic


names = {True: "PCA subset", False: "All candidates"}
r_main, ic_main = oos_summary(fc)
r_alt, ic_alt = oos_summary(fc_alt)
comp = pd.concat({("RMSE ratio vs RW", names[cfg.use_pca]): r_main, ("RMSE ratio vs RW", names[alt_cfg.use_pca]): r_alt,
                  ("IC", names[cfg.use_pca]): ic_main, ("IC", names[alt_cfg.use_pca]): ic_alt}, axis=1).T.round(3)
comp.to_csv(cfg.output_dir / "robustness_feature_selection.csv")
display(comp)

### 5.5 Economic value

The holding period matches the forecast horizon: every 20 trading days, take the position
$\text{sign}(\hat y_{t,20})\times\min(2,\ 25\%/\sigma^{20d}_t)$ and hold it for 20 days ($\sigma^{20d}$: 20-day realized volatility).
A rule driven by the 1-day forecast would trade on day-to-day noise and pay costs almost every day.
Costs of 5 bp per unit of change in the position. Benchmarks: buy-and-hold and a long-only position with the same volatility target and rebalancing dates.

In [ ]:
H_TRADE = cfg.trade_h
d1 = fc[(fc["h"] == 1) & fc["sample"].isin(["Out-of-sample", "Live test"])].dropna(subset=["y"]).set_index("origin")["y"]
sig = fc[(fc["h"] == H_TRADE) & fc["sample"].isin(["Out-of-sample", "Live test", "straddles test start"])].set_index("origin")["XGB"]
sig = sig.reindex(d1.index)
rebal = d1.index[::H_TRADE]                                     # rebalancing dates, every 20 trading days
hold = lambda s: s.reindex(rebal).reindex(d1.index).ffill()    # keep the position between rebalancing dates
lev = np.minimum(cfg.max_leverage, cfg.target_vol / (100 * VOL["rv20"].reindex(d1.index)))
strategies = {"XGBoost signal": hold(np.sign(sig) * lev), "Long, vol-targeted": hold(lev),
              "Buy and hold": pd.Series(1.0, index=d1.index)}
rows_e, curves = [], {}
for name, pos in strategies.items():
    pos = pos.fillna(0.0)
    turn = pos.diff().abs().fillna(pos.abs())
    pnl = pos * d1 - cfg.cost_bps / 1e4 * turn
    cum = pnl.cumsum()
    curves[name] = cum
    rows_e.append({"strategy": name, "annual return %": 100 * 252 * pnl.mean(),
                   "annual vol %": 100 * np.sqrt(252) * pnl.std(), "Sharpe": np.sqrt(252) * pnl.mean() / pnl.std(),
                   "max drawdown %": 100 * (np.exp((cum - cum.cummax()).min()) - 1),
                   "turnover per year": 252 * turn.mean(),
                   "hit rate of 20-day calls %": 100 * ((np.sign(pos.reindex(rebal)) * d1.rolling(H_TRADE).sum().shift(-H_TRADE + 1).reindex(rebal)) > 0).mean()
                   if name == "XGBoost signal" else np.nan})
econ = pd.DataFrame(rows_e).set_index("strategy").round(2)
econ.to_csv(cfg.output_dir / "economic_value.csv")
display(econ)
ax = pd.DataFrame(curves).plot(figsize=(11, 3.5), lw=0.9)
ax.set_title(f"Cumulative log return, {H_TRADE}-day holding period (out-of-sample, after costs)", fontsize=11)
plt.tight_layout(); plt.savefig(cfg.output_dir / "equity_curves.png", dpi=150); plt.show()

## 6. Latest forecast

Re-trained on the most recent window for each horizon, forecast from the last observed close.
Each run appends to `forecast_log.csv`, which is the record for the live test period.

In [ ]:
N = len(X)
P0, d0 = price.iloc[-1], price.index[-1]
live_rows = []
sel_live = pca_select(A_all[N - cfg.window:N], X.columns, cfg.pca_min_eig)[0] if cfg.use_pca else list(range(X.shape[1]))
A_live = A_all[:, sel_live]
for h in cfg.horizons:
    scale = 100 * VOL["rv20"].to_numpy() * np.sqrt(h / 252)
    m, q, _, _ = fit_one(A_live, Y[h].to_numpy(), scale, N - 1, h, cfg)
    mu = cfg.shrink * float(m.predict(A_live[N - 1:N])[0]) / 100
    q = q * scale[N - 1] / 100
    live_rows.append({"asof": str(d0.date()), "h": h, "target_date": (d0 + pd.offsets.BDay(h)).date(),
                      "price_now": P0, "forecast_price": P0 * np.exp(mu),
                      "lo95": P0 * np.exp(mu + q[0]), "lo80": P0 * np.exp(mu + q[1]),
                      "hi80": P0 * np.exp(mu + q[2]), "hi95": P0 * np.exp(mu + q[3])})
live = pd.DataFrame(live_rows)
live.to_csv(cfg.output_dir / "live_forecast.csv", index=False)
log_path = cfg.output_dir / "forecast_log.csv"
flog = pd.concat([pd.read_csv(log_path), live]) if log_path.exists() else live
flog.drop_duplicates(["asof", "h"], keep="last").to_csv(log_path, index=False)
display(live[live["h"].isin(cfg.report_h)].set_index("h").round(2))

fig, ax = plt.subplots(figsize=(11, 4))
hist = price.iloc[-120:]
td = [d0] + [pd.Timestamp(t) for t in live["target_date"]]
col = lambda c: [P0] + list(live[c])
ax.plot(hist.index, hist, color="black", lw=1, label="Brent")
ax.plot(td, col("forecast_price"), color="#1F7A5A", lw=1.5, marker="o", ms=3, label="forecast")
ax.fill_between(td, col("lo80"), col("hi80"), color="#1F7A5A", alpha=0.25, label="80%")
ax.fill_between(td, col("lo95"), col("hi95"), color="#1F7A5A", alpha=0.12, label="95%")
ax.set_title(f"XGBoost forecast from {d0.date()}", fontsize=11)
ax.set_ylabel("$/bbl"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig(cfg.output_dir / "fan_chart.png", dpi=150); plt.show()

## 7. Takeaways

In [ ]:
o_x = xm[xm["sample"] == "Out-of-sample"].set_index("h")
i_x = xm[xm["sample"] == "In-sample"].set_index("h")
wins = int((o_x["RMSE_ratio_vs_RW"] < 1).sum())
sig_h = list(o_x.index[o_x["DM_p"] < 0.05])
top_hi = by_regime["high volatility"].idxmax()
top_lo = by_regime["low volatility"].idxmax()
lines = [
    f"Rolling window {cfg.window} days, re-trained every {cfg.refit_every} days; PCA keeps a median of "
    f"{int(sel_log.sum(axis=1).median())} of {X.shape[1]} candidate features (latest window: {', '.join(X.columns[sel_live])}); "
    f"median trees kept by early stopping: {int(ntrees['trees'].median())}.",
    f"Out-of-sample, XGBoost has lower price RMSE than RW at {wins} of {len(o_x)} horizons "
    f"(ratios {o_x['RMSE_ratio_vs_RW'].min():.3f} to {o_x['RMSE_ratio_vs_RW'].max():.3f}); "
    + (f"significant at 5% (DM) at h = {sig_h}." if sig_h else "none is significant at 5% (DM)."),
    f"In-sample ratios range from {i_x['RMSE_ratio_vs_RW'].min():.3f} to {i_x['RMSE_ratio_vs_RW'].max():.3f}. "
    "The gap to the out-of-sample ratios shows how much of the training fit does not carry over to new data.",
    f"Most important feature of the {cfg.plot_h}-day model: `{share.index[0]}`. "
    f"Top feature by |SHAP| on high-volatility days: `{top_hi}`; on low-volatility days: `{top_lo}`.",
    f"Out-of-sample 95% interval coverage at 60 days: {o_x.loc[60, 'cover95_%']:.0f}%.",
]
display(Markdown("\n".join("- " + s for s in lines)))

## 8. Look-ahead check (optional)

Scrambles all data after a cut-off date, re-runs a short backtest, and checks that every forecast made on or before the cut-off is unchanged.

In [ ]:
if RUN_LEAK_CHECK:
    cut, end = pd.Timestamp("2021-06-30"), "2022-06-30"
    c2 = replace(cfg, oos_start="2020-06-01", horizons=(1, 20))
    rng = np.random.default_rng(0)
    p2, o2 = price.copy(), other.copy()
    after = p2.index > cut
    p2[after] = p2[after] * np.exp(rng.normal(0, 0.05, after.sum()))
    o2.loc[o2.index > cut] = o2.loc[o2.index > cut] * np.exp(rng.normal(0, 0.05, o2.loc[o2.index > cut].shape))
    res = []
    for pp_, oo_ in ((price, other), (p2, o2)):
        Xc, Yc, lc, vc = build_features(pp_.loc[:end], oo_.loc[:end], c2)
        res.append(run_backtest(Xc, Yc, lc, vc, c2, verbose=False)[0])
    k = int(Xc.index.searchsorted(cut, side="right"))
    before = max(np.nanmax(np.abs(res[0][h][m][:k] - res[1][h][m][:k])) for h in c2.horizons
                 for m in ("XGB", "q975", "AR", "HistMean"))
    later = np.nanmax(np.abs(res[0][20]["XGB"][k:] - res[1][20]["XGB"][k:]))
    print(f"max change on/before cut-off: {before:.2e}   after: {later:.2e}")
    print("PASS: no look-ahead" if before < 1e-9 and later > 0 else "FAIL")